In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Assume your text file is named 'input.txt' and is in the same directory
# If not, provide the correct path to the file
file_path = '/content/AP.csv'
lines = []

try:
    with open(file_path, 'r') as f:
        for line in f:
            lines.append(line.strip())  # Remove leading/trailing whitespace and newlines
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    lines = [] # Initialize as empty list if file not found

if lines:
    # Initialize TF-IDF Vectorizer
    # You can adjust parameters like stop_words, max_features, etc.
    vectorizer = TfidfVectorizer()

    # Fit and transform the lines
    tfidf_matrix = vectorizer.fit_transform(lines)

    # Display the results (optional, depending on what you want to do next)
    print("TF-IDF vectorization complete.")
else:
    print("No lines to process.")

TF-IDF vectorization complete.


In [ ]:
from sklearn.cluster import KMeans

# You need to choose the number of clusters (k)
# For example, let's choose 3 clusters. You might need to experiment with this value.
n_clusters = 5

# Initialize and fit KMeans
# Using n_init='auto' to suppress the warning about the default number of initializations
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
kmeans.fit(tfidf_matrix)

# Get the cluster labels for each line
cluster_labels = kmeans.labels_

# Display the cluster labels
print("Cluster labels for each line:")
print(cluster_labels)
print(cluster_labels.shape)


Cluster labels for each line:
[0 3 0 ... 2 0 2]
(1951,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict

# Assuming you have `tfidf_matrix` and `cluster_labels` from previous steps

# Iterate through each cluster
for cluster_id in np.unique(cluster_labels):
    print(f"\nAnalyzing duplicates in Cluster {cluster_id}...")

    # Get the indices of data points in the current cluster
    cluster_indices = np.where(cluster_labels == cluster_id)[0]

    if len(cluster_indices) > 1:
        # Extract the TF-IDF vectors for the current cluster
        cluster_tfidf_matrix = tfidf_matrix[cluster_indices]

        # Calculate the cosine similarity matrix for the current cluster
        cluster_cosine_sim_matrix = cosine_similarity(cluster_tfidf_matrix)




In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict

# Assuming you have `tfidf_matrix` and `cluster_labels` from previous steps

total_duplicates_across_all_clusters = 0

# Iterate through each cluster
for cluster_id in np.unique(cluster_labels):
    print(f"\nAnalyzing duplicates in Cluster {cluster_id}...")

    # Get the indices of data points in the current cluster
    cluster_indices = np.where(cluster_labels == cluster_id)[0]

    if len(cluster_indices) > 1:
        # Extract the TF-IDF vectors for the current cluster
        cluster_tfidf_matrix = tfidf_matrix[cluster_indices]

        # Calculate the cosine similarity matrix for the current cluster
        cluster_cosine_sim_matrix = cosine_similarity(cluster_tfidf_matrix)

        # Dictionary to store groups of duplicate indices (original indices)
        duplicate_groups = {}
        threshold = 1.0

        # Iterate through the upper triangle of the within-cluster cosine similarity matrix
        for i in range(cluster_cosine_sim_matrix.shape[0]):
            for j in range(i + 1, cluster_cosine_sim_matrix.shape[1]):
                if cluster_cosine_sim_matrix[i, j] == threshold:
                    # Get the original indices of the duplicate pair
                    original_index_i = cluster_indices[i]
                    original_index_j = cluster_indices[j]

                    # Check if either index is already in a duplicate group
                    found_group = None
                    for group_id, group in duplicate_groups.items():
                        if original_index_i in group or original_index_j in group:
                            found_group = group_id
                            break

                    if found_group is not None:
                        # Add both indices to the existing group
                        duplicate_groups[found_group].add(original_index_i)
                        duplicate_groups[found_group].add(original_index_j)
                    else:
                        # Create a new group with both indices
                        # Use a unique identifier for the group (e.g., the first index)
                        duplicate_groups[original_index_i] = {original_index_i, original_index_j}

        # Now, process the duplicate_groups to count the number of duplicates in each group
        print(f"Duplicate groups (similarity 1.0) in Cluster {cluster_id}:")
        cluster_total_duplicates = 0
        if duplicate_groups:
            for group_id, group in duplicate_groups.items():
                # The number of duplicates is the size of the group minus 1 (the original)
                num_duplicates = len(group) - 1
                # Only report groups with actual duplicates (size > 1)
                if num_duplicates > 0:
                    print(f"  Group starting with index {group_id}: {num_duplicates} duplicate(s) - Indices: {sorted(list(group))}")
                    cluster_total_duplicates += num_duplicates
            print(f"\nTotal number of unique duplicate groups (similarity 1.0) in Cluster {cluster_id}: {len(duplicate_groups)}")
            print(f"Total duplicates in Cluster {cluster_id}: {cluster_total_duplicates}")
            total_duplicates_across_all_clusters += cluster_total_duplicates

        else:
            print("  No exact duplicate groups found.")


    else:
        print(f"Cluster {cluster_id} has only one data point. No duplicates possible.")

print(f"\nTotal number of duplicates across all unique duplicate groups in all clusters: {total_duplicates_across_all_clusters}")


Analyzing duplicates in Cluster 0...
Duplicate groups (similarity 1.0) in Cluster 0:
  Group starting with index 2: 24 duplicate(s) - Indices: [np.int64(2), np.int64(24), np.int64(29), np.int64(64), np.int64(133), np.int64(180), np.int64(339), np.int64(407), np.int64(487), np.int64(536), np.int64(643), np.int64(889), np.int64(939), np.int64(1010), np.int64(1078), np.int64(1137), np.int64(1332), np.int64(1507), np.int64(1585), np.int64(1699), np.int64(1700), np.int64(1728), np.int64(1729), np.int64(1862), np.int64(1865)]
  Group starting with index 11: 24 duplicate(s) - Indices: [np.int64(11), np.int64(28), np.int64(87), np.int64(110), np.int64(208), np.int64(252), np.int64(301), np.int64(327), np.int64(329), np.int64(370), np.int64(474), np.int64(598), np.int64(611), np.int64(704), np.int64(743), np.int64(765), np.int64(1004), np.int64(1032), np.int64(1041), np.int64(1083), np.int64(1175), np.int64(1208), np.int64(1211), np.int64(1393), np.int64(1447)]
  Group starting with index 41: 

Total number of lines in the dataset: 1951
Total number of duplicates identified: 960
Estimated number of unique lines: 991
Estimated duplication factor (Total Lines / Unique Lines): 1.97
